In [1]:
!pip install transformers torch datasets -q

In [2]:
import torch
from transformers import pipeline

torch.manual_seed(42)
print('Torch version:', torch.__version__)

Torch version: 2.9.0+cu126


## Toy Self-Attention (single head)
Small 3-token sequence with dim=4 to illustrate Q/K/V and attention weights.

In [4]:
# Toy embeddings for 3 tokens
X = torch.tensor([[1., 0., 1., 0.],
                  [0., 2., 0., 2.],
                  [1., 1., 0., 0.]])  # shape (seq, dim)

# Random weight matrices (fixed seed above)
W_q = torch.randn(4, 4)
W_k = torch.randn(4, 4)
W_v = torch.randn(4, 4)

Q = X @ W_q
K = X @ W_k
V = X @ W_v

dk = K.size(-1)
scores = (Q @ K.T) / dk**0.5
weights = scores.softmax(dim=-1)
attended = weights @ V

print('Q shape:', Q.shape)
print('Attention weights (row = query token):')
print(weights)
print('Output embeddings after attention:')
print(attended)

Q shape: torch.Size([3, 4])
Attention weights (row = query token):
tensor([[8.8965e-01, 1.0950e-03, 1.0925e-01],
        [4.8729e-01, 2.3797e-03, 5.1033e-01],
        [9.6236e-01, 6.1762e-06, 3.7635e-02]])
Output embeddings after attention:
tensor([[-2.7361, -0.0272, -0.9745,  1.0933],
        [-1.9822, -0.5926, -0.4999,  1.0190],
        [-2.8716,  0.0710, -1.0585,  1.1097]])


## Pipeline: Sentiment Analysis (DistilBERT)

In [5]:
sentiment = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
texts = [
    'Transformers make NLP much easier.',
    'I am not sure this movie was worth my time.',
    'The food was decent but the service was slow.'
]
for t in texts:
    print(t, '->', sentiment(t)[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


Transformers make NLP much easier. -> {'label': 'POSITIVE', 'score': 0.9677048921585083}
I am not sure this movie was worth my time. -> {'label': 'NEGATIVE', 'score': 0.9989421963691711}
The food was decent but the service was slow. -> {'label': 'NEGATIVE', 'score': 0.9977918863296509}


## Pipeline: Masked Language Modeling (RoBERTa)

In [7]:
fill_mask = pipeline('fill-mask', model='distilroberta-base')
masked_sentence = 'Transformers are <mask> for many NLP tasks.'
for pred in fill_mask(masked_sentence)[:5]:
    print(f"{pred['sequence']} (score={pred['score']:.4f})")

Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Transformers are useful for many NLP tasks. (score=0.1405)
Transformers are suitable for many NLP tasks. (score=0.1381)
Transformers are used for many NLP tasks. (score=0.0520)
Transformers are required for many NLP tasks. (score=0.0477)
Transformers are essential for many NLP tasks. (score=0.0411)


## Pipeline: Text Generation (DistilGPT2)

In [8]:
generator = pipeline('text-generation', model='distilgpt2')
prompt = 'In 2025, natural language models'
outputs = generator(prompt, max_length=40, num_return_sequences=2, do_sample=True, top_p=0.95, top_k=50)
for i, out in enumerate(outputs, 1):
    print(f"\nSample {i}: {out['generated_text']}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 1: In 2025, natural language models for the use of language will be used in a variety of projects, including the development of a series of online educational tools for social media, and the creation of a global network of learning spaces for children.
























































































































































































































Sample 2: In 2025, natural language models will be available in more than 200 countries.
